# 01 - Segment lifts and resample (EMG + IMU)

For each modality, read per-subject raw/filtered CSVs, identify individual **lifts** via contiguous runs of `Activity == 'Lifting-Lowering'` within a `(Participant, Trial Number)` session, drop fatigue trials, extract the channel arrays, and resample each lift to a fixed length (400 timesteps). Outputs compact `.npz` tensor files used by subsequent VAE/DVAE notebooks.

**Outputs (saved to `VAE/Tensors/`):**
- `lifts_emg.npz`  -> `X` shape `(N, 400, 8)` using filtered MVC channels `EMG_MVC_CH1..8`
- `lifts_imu.npz`  -> `X` shape `(N, 400, 96)` using 48 acceleration + 48 segment-rotation channels

Each `.npz` also stores metadata arrays: `participant`, `sex`, `box_mass`, `trial_number`, `lift_idx`, `original_length`, `duration_sec`, `channel_names`.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

ROOT = Path(r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project")
EMG_FILTERED_DIR = ROOT / "EMG" / "Filtered_40"
IMU_RAW_DIR      = ROOT / "IMU" / "Raw"
OUT_DIR          = ROOT / "VAE" / "Tensors"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_LENGTH = 400
EMG_FS_DEFAULT = 500.0
IMU_FS_DEFAULT = 200.0

EMG_CHANNELS = [f"EMG_MVC_CH{i}" for i in range(1, 9)]

META_COLS = {
    "participant":  "Participant",
    "sex":          "Sex",
    "box_mass":     "Box Mass",
    "trial_number": "Trial Number",
    "activity":     "Activity",
    "timestamp":    "Timestamp",
}

In [2]:
def select_imu_channels(columns):
    """Return (accel_cols, rot_cols) in deterministic order.
    Accel: any column containing 'Accel Sensor' (48 expected).
    Rot: any column ending with ' Rot X', ' Rot Y', or ' Rot Z' (48 expected).
    Joint angles (Flexion / Abduction / Axial / etc.) are excluded.
    """
    cols = list(columns)
    accel_cols = [c for c in cols if "Accel Sensor" in c]
    rot_cols   = [c for c in cols if c.endswith((" Rot X", " Rot Y", " Rot Z"))]
    return accel_cols, rot_cols

def find_lift_spans(mask):
    """Given a 1D boolean array, return list of (start, end) for each contiguous True run.
    End is exclusive.
    """
    spans = []
    n = len(mask)
    i = 0
    while i < n:
        if mask[i]:
            j = i + 1
            while j < n and mask[j]:
                j += 1
            spans.append((i, j))
            i = j
        else:
            i += 1
    return spans

def resample_to_length(x2d, target_length):
    """Linear-interpolate each channel to target_length. x2d: (T, C)."""
    T, C = x2d.shape
    if T == target_length:
        return x2d.astype(np.float32, copy=False)
    old = np.linspace(0.0, 1.0, T)
    new = np.linspace(0.0, 1.0, target_length)
    out = np.empty((target_length, C), dtype=np.float32)
    for c in range(C):
        out[:, c] = np.interp(new, old, x2d[:, c])
    return out

def estimate_duration_sec(df_slice, fs_default):
    """Estimate lift duration from row count / sampling rate."""
    n = len(df_slice)
    return float(n) / float(fs_default)

def normalize_sex(x):
    if pd.isna(x):
        return "Unknown"
    s = str(x).strip().lower()
    if s in {"m", "male"}:   return "Male"
    if s in {"f", "female"}: return "Female"
    return str(x)

In [3]:
def segment_file(path, channel_cols, fs_default, target_length):
    """Read one participant CSV, segment into lifts, resample each lift, return (list_of_tensors, list_of_meta)."""
    df = pd.read_csv(path)
    required = list(META_COLS.values())
    for c in required:
        if c not in df.columns:
            raise ValueError(f"{path.name}: missing required column '{c}'")
    for c in channel_cols:
        if c not in df.columns:
            raise ValueError(f"{path.name}: missing channel column '{c}'")

    df["Sex"] = df["Sex"].map(normalize_sex)

    # drop fatigue trials
    fatigue_mask = df["Trial Number"].astype(str).str.contains("fatigue", case=False, na=False)
    df = df.loc[~fatigue_mask].reset_index(drop=True)
    if len(df) == 0:
        return [], []

    tensors, metas = [], []
    for (pid, trial_num), session in df.groupby(["Participant", "Trial Number"], sort=False):
        session = session.reset_index(drop=True)
        activity = session["Activity"].astype(str).str.strip()
        mask = (activity == "Lifting-Lowering").values
        spans = find_lift_spans(mask)
        for lift_idx, (s, e) in enumerate(spans):
            seg = session.iloc[s:e]
            if len(seg) < 10:   # skip implausibly short lifts
                continue
            x = seg[channel_cols].to_numpy(dtype=np.float32)
            x_rs = resample_to_length(x, target_length)
            tensors.append(x_rs)
            metas.append({
                "participant":    str(pid),
                "sex":            str(seg["Sex"].iloc[0]),
                "box_mass":       float(seg["Box Mass"].iloc[0]),
                "trial_number":   str(trial_num),
                "lift_idx":       int(lift_idx),
                "original_length": int(len(seg)),
                "duration_sec":   estimate_duration_sec(seg, fs_default),
            })
    return tensors, metas


def build_modality_tensor(csv_files, channel_cols, fs_default, target_length, label):
    all_X, all_meta = [], []
    for i, path in enumerate(csv_files, 1):
        try:
            tensors, metas = segment_file(path, channel_cols, fs_default, target_length)
        except Exception as e:
            print(f"[{label} {i}/{len(csv_files)}] {path.name} -> FAIL: {e}")
            continue
        all_X.extend(tensors)
        all_meta.extend(metas)
        print(f"[{label} {i}/{len(csv_files)}] {path.name} -> {len(tensors)} lifts")

    if len(all_X) == 0:
        raise RuntimeError(f"No lifts extracted for {label}")
    X = np.stack(all_X, axis=0).astype(np.float32)  # (N, T, C)
    meta_df = pd.DataFrame(all_meta)
    return X, meta_df

In [4]:
# ---------- EMG ----------
emg_files = sorted(EMG_FILTERED_DIR.glob("P*_MVC_envelope.csv"))
print(f"Found {len(emg_files)} filtered EMG files.")

X_emg, meta_emg = build_modality_tensor(
    csv_files=emg_files,
    channel_cols=EMG_CHANNELS,
    fs_default=EMG_FS_DEFAULT,
    target_length=TARGET_LENGTH,
    label="EMG",
)
print("\nEMG tensor shape:", X_emg.shape)
print("EMG lifts per sex:")
print(meta_emg["sex"].value_counts())
print("\nEMG lifts per box_mass:")
print(meta_emg["box_mass"].value_counts().sort_index())

emg_out = OUT_DIR / "lifts_emg.npz"
np.savez_compressed(
    emg_out,
    X=X_emg,
    participant=meta_emg["participant"].to_numpy(),
    sex=meta_emg["sex"].to_numpy(),
    box_mass=meta_emg["box_mass"].to_numpy(dtype=np.float32),
    trial_number=meta_emg["trial_number"].to_numpy(),
    lift_idx=meta_emg["lift_idx"].to_numpy(dtype=np.int32),
    original_length=meta_emg["original_length"].to_numpy(dtype=np.int32),
    duration_sec=meta_emg["duration_sec"].to_numpy(dtype=np.float32),
    channel_names=np.array(EMG_CHANNELS),
)
print("Saved:", emg_out)

Found 40 filtered EMG files.
[EMG 1/40] P10_EMG_MVC_envelope.csv -> 120 lifts
[EMG 2/40] P11_EMG_MVC_envelope.csv -> 120 lifts
[EMG 3/40] P12_EMG_MVC_envelope.csv -> 84 lifts
[EMG 4/40] P13_EMG_MVC_envelope.csv -> 120 lifts
[EMG 5/40] P14_EMG_MVC_envelope.csv -> 120 lifts
[EMG 6/40] P15_EMG_MVC_envelope.csv -> 120 lifts
[EMG 7/40] P16_EMG_MVC_envelope.csv -> 120 lifts
[EMG 8/40] P17_EMG_MVC_envelope.csv -> 120 lifts
[EMG 9/40] P18_EMG_MVC_envelope.csv -> 120 lifts
[EMG 10/40] P19_EMG_MVC_envelope.csv -> 108 lifts
[EMG 11/40] P1_EMG_MVC_envelope.csv -> 119 lifts
[EMG 12/40] P20_EMG_MVC_envelope.csv -> 120 lifts
[EMG 13/40] P21_EMG_MVC_envelope.csv -> 120 lifts
[EMG 14/40] P22_EMG_MVC_envelope.csv -> 120 lifts
[EMG 15/40] P23_EMG_MVC_envelope.csv -> 120 lifts
[EMG 16/40] P24_EMG_MVC_envelope.csv -> 120 lifts
[EMG 17/40] P25_EMG_MVC_envelope.csv -> 120 lifts
[EMG 18/40] P26_EMG_MVC_envelope.csv -> 120 lifts
[EMG 19/40] P27_EMG_MVC_envelope.csv -> 120 lifts
[EMG 20/40] P28_EMG_MVC_envelope

In [5]:
# ---------- IMU ----------
imu_files = sorted(IMU_RAW_DIR.glob("P*_IMU.csv"))
print(f"Found {len(imu_files)} IMU files.")

# determine IMU channel order from the first file
probe_cols = pd.read_csv(imu_files[0], nrows=1).columns
accel_cols, rot_cols = select_imu_channels(probe_cols)
IMU_CHANNELS = accel_cols + rot_cols
print(f"IMU channels: {len(accel_cols)} accel + {len(rot_cols)} rot = {len(IMU_CHANNELS)}")
assert len(IMU_CHANNELS) == 96, f"Expected 96 IMU channels, got {len(IMU_CHANNELS)}"

X_imu, meta_imu = build_modality_tensor(
    csv_files=imu_files,
    channel_cols=IMU_CHANNELS,
    fs_default=IMU_FS_DEFAULT,
    target_length=TARGET_LENGTH,
    label="IMU",
)
print("\nIMU tensor shape:", X_imu.shape)
print("IMU lifts per sex:")
print(meta_imu["sex"].value_counts())
print("\nIMU lifts per box_mass:")
print(meta_imu["box_mass"].value_counts().sort_index())

imu_out = OUT_DIR / "lifts_imu.npz"
np.savez_compressed(
    imu_out,
    X=X_imu,
    participant=meta_imu["participant"].to_numpy(),
    sex=meta_imu["sex"].to_numpy(),
    box_mass=meta_imu["box_mass"].to_numpy(dtype=np.float32),
    trial_number=meta_imu["trial_number"].to_numpy(),
    lift_idx=meta_imu["lift_idx"].to_numpy(dtype=np.int32),
    original_length=meta_imu["original_length"].to_numpy(dtype=np.int32),
    duration_sec=meta_imu["duration_sec"].to_numpy(dtype=np.float32),
    channel_names=np.array(IMU_CHANNELS),
)
print("Saved:", imu_out)

Found 40 IMU files.
IMU channels: 48 accel + 48 rot = 96


C:\Users\lilin\AppData\Local\Temp\ipykernel_60152\1782808707.py:3: DtypeWarning: Columns (135,136,137,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


[IMU 1/40] P10_IMU.csv -> 120 lifts


C:\Users\lilin\AppData\Local\Temp\ipykernel_60152\1782808707.py:3: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


[IMU 2/40] P11_IMU.csv -> 120 lifts
[IMU 3/40] P12_IMU.csv -> 120 lifts


...

[IMU 7/40] P16_IMU.csv -> 120 lifts
[IMU 8/40] P17_IMU.csv -> 120 lifts
[IMU 9/40] P18_IMU.csv -> 120 lifts
[IMU 10/40] P19_IMU.csv -> 120 lifts
[IMU 11/40] P1_IMU.csv -> 120 lifts
[IMU 12/40] P20_IMU.csv -> 120 lifts
[IMU 13/40] P21_IMU.csv -> 120 lifts
[IMU 14/40] P22_IMU.csv -> 120 lifts
[IMU 15/40] P23_IMU.csv -> 120 lifts
[IMU 16/40] P24_IMU.csv -> 120 lifts
[IMU 17/40] P25_IMU.csv -> 120 lifts
[IMU 18/40] P26_IMU.csv -> 120 lifts
[IMU 19/40] P27_IMU.csv -> 120 lifts
[IMU 20/40] P28_IMU.csv -> 120 lifts
[IMU 21/40] P29_IMU.csv -> 120 lifts
[IMU 22/40] P2_IMU.csv -> 120 lifts
[IMU 23/40] P30_IMU.csv -> 120 lifts
[IMU 24/40] P31_IMU.csv -> 120 lifts
[IMU 25/40] P32_IMU.csv -> 120 lifts
[IMU 26/40] P33_IMU.csv -> 120 lifts
[IMU 27/40] P34_IMU.csv -> 120 lifts
[IMU 28/40] P35_IMU.csv -> 120 lifts
[IMU 29/40] P36_IMU.csv -> 120 lifts
[IMU 30/40] P37_IMU.csv -> 120 lifts
[IMU 31/40] P38_IMU.csv -> 120 lifts
[IMU 32/40] P39_IMU.csv -> 120 lifts


C:\Users\lilin\AppData\Local\Temp\ipykernel_60152\1782808707.py:3: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


[IMU 33/40] P3_IMU.csv -> 120 lifts
[IMU 34/40] P40_IMU.csv -> 120 lifts
[IMU 35/40] P4_IMU.csv -> 120 lifts


C:\Users\lilin\AppData\Local\Temp\ipykernel_60152\1782808707.py:3: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


[IMU 36/40] P5_IMU.csv -> 120 lifts
[IMU 37/40] P6_IMU.csv -> 120 lifts
[IMU 38/40] P7_IMU.csv -> 120 lifts
[IMU 39/40] P8_IMU.csv -> 120 lifts
[IMU 40/40] P9_IMU.csv -> 120 lifts

IMU tensor shape: (4800, 400, 96)
IMU lifts per sex:
sex
Female    2400
Male      2400
Name: count, dtype: int64

IMU lifts per box_mass:
box_mass
2.3     960
4.5     960
6.8     960
9.1     960
11.3    960
Name: count, dtype: int64
Saved: C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\VAE\Tensors\lifts_imu.npz


In [6]:
# ---------- sanity checks ----------
def summarize(npz_path, label):
    data = np.load(npz_path, allow_pickle=True)
    X = data["X"]
    print(f"\n=== {label}: {npz_path.name} ===")
    print("X shape:", X.shape, "dtype:", X.dtype)
    print("N subjects:", len(np.unique(data["participant"])))
    print("Duration (s): min={:.2f}, median={:.2f}, max={:.2f}".format(
        float(data["duration_sec"].min()),
        float(np.median(data["duration_sec"])),
        float(data["duration_sec"].max()),
    ))
    print("Original length: min={}, median={:.0f}, max={}".format(
        int(data["original_length"].min()),
        float(np.median(data["original_length"])),
        int(data["original_length"].max()),
    ))
    print("Lifts per box_mass:")
    print(pd.Series(data["box_mass"]).value_counts().sort_index())
    print("Lifts per (sex, box_mass):")
    print(pd.crosstab(pd.Series(data["sex"]), pd.Series(data["box_mass"])))

summarize(OUT_DIR / "lifts_emg.npz", "EMG")
summarize(OUT_DIR / "lifts_imu.npz", "IMU")


=== EMG: lifts_emg.npz ===
X shape: (4750, 400, 8) dtype: float32
N subjects: 40
Duration (s): min=2.00, median=6.00, max=32.00
Original length: min=1000, median=3000, max=16000
Lifts per box_mass:
2.3     947
4.5     960
6.8     947
9.1     948
11.3    948
Name: count, dtype: int64
Lifts per (sex, box_mass):
col_0   2.3   4.5   6.8   9.1   11.3
row_0                               
Female   468   480   467   468   468
Male     479   480   480   480   480

=== IMU: lifts_imu.npz ===
X shape: (4800, 400, 96) dtype: float32
N subjects: 40
Duration (s): min=2.00, median=6.00, max=32.00
Original length: min=400, median=1200, max=6400
Lifts per box_mass:
2.3     960
4.5     960
6.8     960
9.1     960
11.3    960
Name: count, dtype: int64
Lifts per (sex, box_mass):
col_0   2.3   4.5   6.8   9.1   11.3
row_0                               
Female   480   480   480   480   480
Male     480   480   480   480   480
